# 04 -- Prediction adjudication

Narrative only (`project.md` SS9). Runs the actual, real adjudication script
(`scripts/adjudicate_predictions.py`) that decides every verdict in
`paper/sections/results.tex`/`negative_results.tex` and `FINDINGS.md` -- this notebook
does not re-implement or re-derive any of the check logic, it just invokes the real
script and displays its output. CPU-only, reads only committed `results/runs/` data.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "configs").is_dir() else Path.cwd().parent
result = subprocess.run(
    [sys.executable, str(REPO_ROOT / "scripts" / "adjudicate_predictions.py")],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
# a RankWarning on stderr is expected and benign when PR-10's degenerate-fit guard
# fires (see FINDINGS.md F9) -- the real output is the JSON on stdout.
report = json.loads(result.stdout[result.stdout.index("{"):])
print(f"{len(report)} predictions/checks adjudicated")

## Verdict table

In [ ]:
verdicts = {k: v.get("verdict", "?") for k, v in report.items()}
counts = {}
for v in verdicts.values():
    counts[v] = counts.get(v, 0) + 1

print(f"{'prediction':20s} verdict")
for k, v in verdicts.items():
    print(f"{k:20s} {v}")
print()
print("counts by verdict:", counts)
assert "?" not in counts, "an un-adjudicated prediction leaked through -- T4.7's own DoD requires zero of these"

## A specific example: PR-10's honest INSUFFICIENT_DATA

Worth showing in detail: PR-10 (barren-plateau decay rate) does NOT get a fabricated
slope even though the underlying fit function *could* silently return one --
`barren_plateau_fit` (`src/qapinn/xai/gradvar.py`) detects that `depth_sweep`'s cut
grid has zero variation in `n_qubits` (the fit's own independent variable) and refuses
to report a slope, rather than returning `numpy.polyfit`'s ill-conditioned solver
artifact. See `FINDINGS.md` F9.

In [ ]:
pr10 = report["PR-10"]
print(json.dumps(pr10, indent=2))